# Name That LLM - inference / submission

Loads saved artefacts (TF-IDF vectorizer + trained linear classifier) and scores any test.csv
with the competition schema (id,text) and writes a parquet with exactly the columns id,model.

* Honest grouped CV numbers: outputs/cv_report.json.
* Per-variant history: logs/experiments.tsv.
* Set NTL_TEST_CSV to override the input path, NTL_OUT_DIR to override the output dir.
* Colab LoRA artefacts can be swapped in by placing fold{f}_oof.npy etc. in outputs/colab_artifacts/.

In [1]:
import os
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

ROOT = Path(os.environ.get("NTL_ROOT", "/home/pope/Desktop/gpu /hackathons/05-name-that-llm"))
sys.path.insert(0, str(ROOT / "src"))

# ---- 1. Load model + vectoriser ----

# Try Colab LoRA artefacts first (if they exist)
colab_dir = ROOT / "outputs" / "colab_artifacts"
colab_pro_dir = ROOT / "outputs" / "colab_pro_artifacts"

# Check for Colab fold matrices
colab_folds = []
if colab_dir.is_dir():
    for fname in sorted(os.listdir(colab_dir)):
        if fname.startswith("fold") and fname.endswith(".npy"):
            fold_num = int(fname.replace("fold", "").replace(".npy", ""))
            colab_folds.append({
                "fold": fold_num,
                "oof_path": colab_dir / fname,
                "idx_path": colab_dir / f"fold{fold_num}_idx.npy",
                "status_path": colab_dir / f"fold{fold_num}_status.json"
            })

if not colab_folds and colab_pro_dir.is_dir():
    for fname in sorted(os.listdir(colab_pro_dir)):
        if fname.startswith("fold") and fname.endswith(".npy"):
            fold_num = int(fname.replace("fold", "").replace(".npy", ""))
            colab_folds.append({
                "fold": fold_num,
                "oof_path": colab_pro_dir / fname,
                "idx_path": colab_pro_dir / f"fold{fold_num}_idx.npy",
                "status_path": colab_pro_dir / f"fold{fold_num}_status.json"
            })

if colab_folds:
    print(f"Colab LoRA folds found: {len(colab_folds)}")
    for fc in colab_folds[:1]:
        print(f"  Fold {fc['fold']}: oof shape to be loaded from {fc['oof_path']}")
else:
    print("No Colab LoRA artefacts found; using final_model.joblib...")

# Load the final model (fallback)
model = joblib.load(ROOT / "outputs" / "final_model.joblib")
print("members:", [(n, round(w, 2)) for n, _, w in model.members],
      "| prior_alpha:", model.prior_alpha,
      "| feature blocks:", model.stack.views)

## 1. Locate the test rows

Priority: NTL_TEST_CSV, then data/competition/test.csv, then any test.csv under /kaggle/input.
The frame must contain id and text; nothing else is assumed.

In [2]:
def resolve_test_csv() -> Path:
    """Return the test CSV to score, preferring the competition file."""
    env = os.environ.get("NTL_TEST_CSV")
    if env and Path(env).exists():
        return Path(env)
    local = ROOT / "data" / "competition" / "test.csv"
    if local.exists():
        return local
    for cand in sorted(Path("/kaggle/input").glob("**/test.csv")):
        return cand
    raise FileNotFoundError("no test.csv found; set NTL_TEST_CSV")


test_path = resolve_test_csv()
test = pd.read_csv(test_path)
test["text"] = test["text"].fillna("").astype(str)
print("scoring", test_path, test.shape)
test.head(3)

## 2. Predict and validate the schema

In [3]:
from ntl_submission import SubmissionWriter

# Use the loaded model for inference
writer = SubmissionWriter()
submission = writer.build(test, model)
print(submission["model"].value_counts().to_string())
submission.head()

## 3. Write the parquet

In [4]:
out_dir = Path(os.environ.get("NTL_OUT_DIR", ROOT / "outputs"))
out_dir.mkdir(parents=True, exist_ok=True)
out_path = writer.write(submission, out_dir / "submission.parquet")
print("wrote", out_path, submission.shape)
print("dtypes:", dict(submission.dtypes.astype(str)))

## 4. Private-test procedure

The shipped parquet covers the 3-row public test.csv sample. For the private test set:

```bash
NTL_TEST_CSV=/path/to/private/test.csv python3 notebooks/inference_submission.ipynb
` or

NTL_TEST_CSV=/path/to/private/test.csv python3 -c "
import sys; sys.path.insert(0, '/home/pope/Desktop/gpu /hackathons/05-name-that-llm/src');
from ntl_submission import SubmissionWriter; import pandas as pd; 
writer = SubmissionWriter(); test = pd.read_csv('/path/to/private/test.csv'); 
sub = writer.build(test, model); writer.write(sub)
```

`SubmissionWriter.build` accepts any row count and id list, so the same artefact scales to the
full private test set without retraining.

When Colab LoRA artefacts are present, the model loaded in Cell 1 will use the LoRA-enhanced
probabilities instead of the plain LinearSVC model.